# Lesson 2 — Why AI Makes Things Up

### AI Study Buddy · Lesson 2 of 8

Today has no solution in it. Today is about **feeling the problem**, twice, properly — so
that the next three sessions feel necessary instead of arbitrary.

We're going to ask a model about material it has never seen, and it will answer. Confidently.
In detail. With specifics. And every specific will be invented.

Then we'll try the obvious fix — paste the whole document into the prompt — and that will
work, right up until it doesn't, for reasons we'll measure rather than guess at.

## TODAY:

- **Part A:** watch a model invent things
- **Part B:** the brute-force fix, and why it works
- **Part C:** why it can't scale — measured, not asserted

In [ ]:
# imports

import os
from pathlib import Path
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
# Setup

load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")
print(f"✅ Key loaded, begins {api_key[:11]}" if api_key else "❌ No key found - check your .env")

CHAT_MODEL = "openai:gpt-5.4-mini"
model = init_chat_model(CHAT_MODEL)

In [ ]:
# The `ask` function from last session, for us to keep using

def ask(system_prompt, question):
    """Ask the model a question with a given system prompt. Returns the reply as a string."""
    response = model.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=question),
    ])
    return response.text

## PART A: Watch a model invent things

In the `documents/` folder next to this notebook there are four files about **the Kestrel
Project** — a student high-altitude balloon programme.

The Kestrel Project does not exist. It was invented for this workshop, it has never been
written about anywhere, and no AI model has ever seen a word of it.

That's exactly why we're using it. If we asked about photosynthesis, the model would know
the answer anyway, and we'd never be able to tell whether anything we built was working.

In [ ]:
# Let's ask about something very specific from the documents.
# You can check the true answer yourself - open documents/flights.md.

question = "What altitude did Kestrel Flight 8 reach, and why is it significant?"

print(ask("You are a helpful assistant.", question))

### Read that again

Go and open `documents/flights.md` and compare.

The model almost certainly gave you a number. It may have given you a date, a team, a
purpose. It probably sounded like someone who knows what they're talking about.

It has never seen this document. Every specific in that answer was generated.

In [ ]:
# Try a few more. Collect the best inventions - we're putting them on the whiteboard.

for q in [
    "Who founded the Kestrel Project?",
    "What is the maximum payload mass allowed on a Kestrel flight?",
    "Why does every Kestrel flight carry two trackers?",
]:
    print(f"Q: {q}")
    print(f"A: {ask('You are a helpful assistant.', q)}\n")
    print("-" * 70)

### Why does it do this?

This is the important idea of the session, and it isn't a bug.

A language model is, underneath, a machine that **predicts what text comes next**. It was
trained on an enormous amount of writing, and it got extremely good at producing text that
*looks like* the kind of text that would follow.

It has no database. It has no list of facts it can look things up in. And crucially, it has
no built-in sense of the difference between *"I am recalling this"* and *"I am composing
something plausible"* — because from the inside, it's doing the same thing either way.

"What altitude did Flight 8 reach?" is, to the model, a request to produce text that looks
like an answer to that question. And it does. Beautifully.

This is usually called **hallucination**, which is a slightly generous word for it.

<div style="border-left: 6px solid #181; background: #f3fbf3; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#181; margin-top:0;">🌍 Where you'll see this in the real world</h3>
<p style="color:#181; margin-bottom:0;">
Lawyers have submitted court filings citing cases that never existed, because a model
invented them and the citations looked right. Students have handed in essays quoting
papers that don't exist. Support chatbots have invented refund policies their companies
then had to honour.<br/><br/>
In every one of those, the failure wasn't that the model was stupid. It's that it was
<b>fluent</b>, and fluency is extremely convincing. The whole of this course is about the
technique that fixes it.
</p>
</div>

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Ask three questions about <b>your own</b> documents — the ones you brought.<br/><br/>
Then ask one question about something that <i>definitely doesn't exist</i>: an invented
book, an invented country, a rule you just made up.<br/><br/>
Does it ever say "I don't know"? How often? And when it does make something up, how would
someone who <i>didn't</i> know the subject be able to tell?
</p>
</div>

In [ ]:
# Your turn

print(ask("You are a helpful assistant.", ""))

## PART B: The brute-force fix

The model doesn't know because we never told it. So — tell it.

In [ ]:
# Read one file. The `encoding="utf-8"` matters: without it, Windows sometimes
# guesses a different encoding and mangles any character that isn't plain English.

flights = Path("documents/flights.md").read_text(encoding="utf-8")

print(flights[:400])

In [ ]:
# How big is it?

f"{len(flights):,} characters"

Now we put that text into the **system prompt** — the message that tells the model what its
job is. The model can read anything in its prompt, so anything we put there, it knows.

In [ ]:
GROUNDED_SYSTEM = """
You are a helpful assistant answering questions about the Kestrel Project.
Use the information below to answer. It is the only information you have.

INFORMATION:
{context}
"""

In [ ]:
# .format() fills in the {context} placeholder.
#
# We're using a CAPITALS template plus .format() rather than an f-string because the
# template needs to exist before we know what goes in it - and because a long prompt is
# much easier to read and edit when it's sitting on its own at the top.

system_prompt = GROUNDED_SYSTEM.format(context=flights)

print(system_prompt[:500])

In [ ]:
# Now ask the same question that failed before

print(ask(system_prompt, "What altitude did Kestrel Flight 8 reach, and why is it significant?"))

### That's it. That's the whole idea.

Compare that answer with the first one in Part A. Same model, same question. The only
difference is that this time the relevant text was sitting in the prompt.

This is called **grounding**: the answer is anchored in text you supplied, rather than
floating free.

In [ ]:
# Try the others too

for q in [
    "Which flight holds the altitude record?",
    "Why did Flight 4 take eleven days to recover?",
    "What happened on Flight 2?",
]:
    print(f"Q: {q}")
    print(f"A: {ask(system_prompt, q)}\n")
    print("-" * 70)

### But watch this

We only loaded `flights.md`. Ask something that's in one of the *other* files:

In [ ]:
print(ask(system_prompt, "What flight computer does Kestrel use?"))

It either said it doesn't know — good! — or it made something up again. Either way, the
information is in `hardware.md`, which we never loaded.

So: load everything.

In [ ]:
# Load every markdown file in the documents folder and glue them together.
#
# sorted() is there so the order is the same every time you run it. Without it, the order
# depends on the filesystem, which makes results that vary for no visible reason - and
# "it worked yesterday" is the worst kind of bug.

everything = ""
for path in sorted(Path("documents").glob("*.md")):
    everything += f"\n\n--- {path.name} ---\n\n"
    everything += path.read_text(encoding="utf-8")

print(f"Loaded {len(everything):,} characters from all files")

In [ ]:
system_prompt = GROUNDED_SYSTEM.format(context=everything)

print(ask(system_prompt, "What flight computer does Kestrel use, and why did they switch to it?"))

<div style="border-left: 6px solid #900; background: #fff4f4; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#900; margin-top:0;">🎯 Now try it yourself</h3>
<p style="color:#900; margin-bottom:0;">
Do the same with <b>your own</b> documents. Load them all, build the system prompt, and ask
the three questions that failed in Part A.<br/><br/>
Then find a question that needs information from <b>two different files</b> at once — and
see whether it manages.
</p>
</div>

In [ ]:
# Your documents. Change the folder name if yours is somewhere else.

my_text = ""
for path in sorted(Path("documents").glob("*.md")):
    my_text += f"\n\n--- {path.name} ---\n\n" + path.read_text(encoding="utf-8")

my_system = GROUNDED_SYSTEM.format(context=my_text)

print(ask(my_system, ""))

## PART C: Why this can't scale

So we've solved it. Paste everything in. Done, three sessions early.

Except: let's measure what we just did.

In [ ]:
# Our whole knowledge base is tiny

f"{len(everything):,} characters"

### Tokens

Models don't read characters, they read **tokens** — chunks of roughly 4 characters, about
¾ of a word. Everything a model charges for, and every limit it has, is measured in tokens.

In [ ]:
# A rough estimate is fine for our purposes: about 4 characters per token.

def estimate_tokens(text):
    """Roughly how many tokens is this text? Good enough for deciding if something will fit."""
    return len(text) // 4


estimate_tokens(everything)

### The three walls you hit

**Wall 1 — the context window.** Every model has a maximum prompt size. For GPT-5.4 mini
it's 400,000 tokens, which is a *lot*. Our four little files don't come close. Now
scale it up, and watch where it actually breaks:

In [ ]:
# What if your knowledge base were bigger? Let's find out where it stops fitting.
#
# CONTEXT_LIMIT is the model's maximum prompt size in tokens. For GPT-5.4 mini that's
# 400,000. A textbook is roughly 500,000 characters; a year of school notes, several million.

CONTEXT_LIMIT = 400_000

for name, characters in [
    ("our 4 Kestrel files", len(everything)),
    ("one textbook chapter", 40_000),
    ("one whole textbook", 500_000),
    ("a year of school notes", 3_000_000),
    ("the English Wikipedia", 20_000_000_000),
]:
    tokens = characters // 4
    fits = "fits ✅" if tokens < CONTEXT_LIMIT else "DOESN'T FIT ❌"
    print(f"{name:25} {tokens:>15,} tokens   {fits}")

### Notice what just happened

**A whole textbook fits.** Context windows have grown enormously — a few years ago 400,000
tokens would have been unthinkable, and plenty of tutorials still tell you the context
window is the reason you need RAG.

For a single textbook, on this model, that reason no longer holds. Be suspicious of
anyone who tells you otherwise without checking the number.

But look one row down: a year of your own school notes still doesn't fit, and a large
company's documents miss by a factor of thousands. The wall didn't disappear — it moved.

And the next two walls never moved at all. They're the ones that actually decide this.

**Wall 2 — cost.** You pay per token, on **every single question**. Not once — every time.

In [ ]:
# Prices change, but the shape of the answer doesn't.
# GPT-5.4 mini charges $0.75 per million tokens you send it.

PRICE_PER_MILLION_TOKENS = 0.75


def cost_per_question(characters):
    """What one question costs if you paste this much text into every prompt."""
    tokens = characters // 4
    return tokens / 1_000_000 * PRICE_PER_MILLION_TOKENS


for name, characters in [
    ("our 4 Kestrel files", len(everything)),
    ("one whole textbook", 500_000),
    ("a year of school notes", 3_000_000),
]:
    cost = cost_per_question(characters)
    print(f"{name:25} ${cost:.5f} per question   →  ${cost * 100:.2f} for 100 questions")

**Wall 3 — and this one surprises people.** A model given an enormous prompt gets **worse**,
not just slower and more expensive. The answer you want is buried in thousands of lines of
things you didn't want, and models are measurably more likely to lose track of information
in the middle of a long context than at either end.

Handing someone an entire library and asking them a question is not more helpful than
handing them the right page. It's less helpful.

So even in the case where the whole textbook *does* fit, you still shouldn't send it.
"It fits" and "it's a good idea" are different questions, and only the first one has a
number you can look up.

<div style="border-left: 6px solid #06c; background: #f2f7ff; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#06c; margin-top:0;">🎯 So what do we actually want?</h3>
<p style="color:#06c; margin-bottom:0;">
Say it out loud before you read on. We want to send <b>only the two or three paragraphs
that are actually relevant to this particular question</b> — not the whole library.<br/><br/>
Which means we need to answer one question, mechanically, before every single query:<br/><br/>
<b>"Which pieces of my documents are relevant to what was just asked?"</b><br/><br/>
That's it. That's the rest of the course. Everything from here — chunking, embeddings,
vector stores, similarity search — exists to answer that one question well.
</p>
</div>

<div style="border-left: 6px solid #f71; background: #fff7f0; padding: 12px 16px; margin: 12px 0;">
<h3 style="color:#f71; margin-top:0;">🚀 Extra challenge</h3>
<p style="color:#f71; margin-bottom:0;">
Try the dumbest possible version of "find the relevant pieces": split the text on blank
lines into paragraphs, and keep only the paragraphs containing a word from the question.<br/><br/>
Then find a question where it works well — and a question where it fails completely.<br/><br/>
The failure is the interesting one. "What's the heaviest thing they're allowed to fly?"
shares no words at all with "No payload exceeds 2.8 kg in total mass." Word matching can't
see that those are about the same thing. <b>That</b> is the problem Lesson 4 solves.
</p>
</div>

In [ ]:
# The dumbest possible retrieval. Try to break it.

def find_paragraphs(text, question):
    """Keep only paragraphs sharing at least one word (longer than 3 letters) with the question."""
    question_words = {w.lower().strip("?,.") for w in question.split() if len(w) > 3}
    paragraphs = text.split("\n\n")
    return [p for p in paragraphs if question_words & {w.lower().strip("?,.") for w in p.split()}]


find_paragraphs(everything, "What is the heaviest payload allowed?")

## Where we got to

- A language model predicts text. It has no database, and no internal sense of the
  difference between remembering and inventing
- Grounding: put the information in the prompt, and the answer becomes anchored to it
- Brute force works on four small files and falls over on three walls — context window,
  cost per question, and degraded quality on long prompts
- The real question, which the next three sessions answer: **which pieces are relevant?**

## Next lesson

Before we can find the relevant pieces, we need pieces. Next session: cutting documents up
properly — and discovering that how you cut them matters more than you'd expect.